# 第七章：PyTorch 可视化

这一章原文讲网络结构、卷积层特征图、TensorBoard、wandb、SwanLab 等。这里先做本地最稳的可运行版本：打印模型结构、抓取中间特征图、写入 TensorBoard 日志，并检测外部可视化工具是否已安装。

## 1. 准备模型和样例图片

我们定义一个很小的 CNN，用随机图片跑通可视化流程。

In [ ]:
from pathlib import Path
import tempfile
import importlib.util

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter

torch.manual_seed(42)

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 4, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(4, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(8, 2)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

model = TinyCNN()
sample = torch.randn(1, 1, 16, 16)
print(model)


## 2. 查看输入和输出形状

可视化第一步不是画图，而是确认每一步的张量形状符合预期。

In [ ]:
with torch.no_grad():
    output = model(sample)

print("输入形状：", sample.shape)
print("输出形状：", output.shape)
print("输出值：", output)


## 3. 用 hook 抓取中间特征图

`register_forward_hook` 可以在某一层前向传播结束时，拿到这一层的输出。

In [ ]:
activations = {}

def save_activation(name):
    def hook(module, inputs, output):
        activations[name] = output.detach()
    return hook

handle = model.features[0].register_forward_hook(save_activation("conv1"))
with torch.no_grad():
    _ = model(sample)
handle.remove()

print("抓到的层：", list(activations.keys()))
print("conv1 特征图形状：", activations["conv1"].shape)


## 4. 可视化原图和卷积特征图

这里的输入是随机图，所以图像没有语义；重点是理解“一个卷积层会输出多个通道的特征图”。

In [ ]:
feature = activations["conv1"][0]

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
axes[0].imshow(sample[0, 0], cmap="gray")
axes[0].set_title("输入")
axes[0].axis("off")

for i in range(4):
    axes[i + 1].imshow(feature[i], cmap="viridis")
    axes[i + 1].set_title(f"特征图 {i}")
    axes[i + 1].axis("off")

plt.tight_layout()
plt.show()


## 5. 可视化卷积核权重

卷积层权重也可以看成小图。第一层卷积核形状通常是 `[输出通道数, 输入通道数, 高, 宽]`。

In [ ]:
weights = model.features[0].weight.detach()
print("第一层卷积核形状：", weights.shape)

fig, axes = plt.subplots(1, 4, figsize=(10, 2.5))
for i in range(4):
    axes[i].imshow(weights[i, 0], cmap="coolwarm")
    axes[i].set_title(f"卷积核 {i}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()


## 6. 写入 TensorBoard 日志

这格会创建一个临时日志目录，写入模型图、输入图片和一个标量。真实项目可以把 `logdir` 改成 `runs/实验名`，然后在终端运行 `tensorboard --logdir runs`。

In [ ]:
with tempfile.TemporaryDirectory() as logdir:
    writer = SummaryWriter(logdir)
    writer.add_graph(model, sample)
    writer.add_image("sample/random_image", sample[0], global_step=0)
    writer.add_scalar("demo/loss", 0.42, global_step=1)
    writer.close()

    event_files = list(Path(logdir).glob("events.out.tfevents.*"))
    print("TensorBoard 日志目录：", logdir)
    print("写入事件文件数量：", len(event_files))
    print("真实项目中可运行：tensorboard --logdir", logdir)


## 7. 检查 wandb 和 SwanLab

这两个工具通常需要账号或外部服务。为了让 notebook 不打断学习，这里只检测是否安装，并说明下一步。

In [ ]:
for package_name in ["wandb", "swanlab"]:
    if importlib.util.find_spec(package_name) is None:
        print(f"{package_name}: 未安装。需要时可用 %pip install {package_name} 安装。")
    else:
        print(f"{package_name}: 已安装，可以继续配置登录和实验记录。")


## 8. 小结

- `print(model)` 是最快的结构查看方式。
- hook 可以抓中间层输出，用于理解 CNN 学到了什么。
- TensorBoard 适合记录训练曲线、图片和模型图。
- wandb/SwanLab 更适合长期实验管理，但通常需要账号配置。